# Chapter 34: Multi-Agent Orchestration

> **What this notebook teaches**: How to build systems where multiple specialized AI agents work together — each expert in one domain — coordinated by a supervisor.

## Why One Agent Is Not Enough

A single agent can troubleshoot a BGP problem. Ask it to simultaneously diagnose routing, check security ACLs, analyze QoS metrics, and generate remediation docs — quality drops. Context fills. The attention mechanism is diluted across too many domains.

**The fix**: specialized agents. Each agent handles one domain. A supervisor coordinates.

## 5 Patterns You Will Build

| Demo | Pattern | What It Solves |
|------|---------|---------------|
| 1 | **Supervisor (Hub-and-Spoke)** | Route alerts to the right specialist |
| 2 | **Actor-Critic** | Self-review config changes before they go live |
| 3 | **Sequential vs Parallel** | Optimize latency based on task dependencies |
| 4 | **Shared State + Handoffs** | Pass context cleanly between agents |
| 5 | **Hallucination Cascade Defense** | Validate between agents to stop error propagation |

> **Network Analogy**: Multi-agent = real NOC shift. Major outage → shift lead (supervisor) dispatches BGP specialist, firewall engineer, change manager simultaneously. Each works their domain. Shift lead synthesizes findings. Same pattern, applied to LLM agents.

**No external frameworks** — pure Python + Anthropic SDK (simulation fallback included).


In [ ]:
import os
import json
import time
import threading
import random
from dataclasses import dataclass, field
from typing import Optional

# ── Anthropic client ──────────────────────────────────────────────────────────
try:
    from anthropic import Anthropic
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

    API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
    if API_KEY:
        client = Anthropic()
        USE_REAL_API = True
        print("✓ Anthropic client ready — real API")
    else:
        USE_REAL_API = False
        print("ℹ No API key — using simulation (all patterns still demonstrated)")
except ImportError:
    USE_REAL_API = False
    print("ℹ anthropic not installed — pip install anthropic")

HAIKU = "claude-haiku-4-5-20251001"

# ── Agent base class ──────────────────────────────────────────────────────────
class Agent:
    """
    Base agent: has a name, a system prompt (role), and can call the LLM.
    Think of this as one specialist engineer in the NOC.
    """
    def __init__(self, name: str, system: str):
        self.name   = name
        self.system = system

    def run(self, task: str, context: str = "") -> str:
        """Execute a task and return a response."""
        prompt = f"{context}\n\nTask: {task}" if context else task
        if USE_REAL_API:
            msg = client.messages.create(
                model=HAIKU,
                max_tokens=500,
                system=self.system,
                messages=[{"role": "user", "content": prompt}]
            )
            return msg.content[0].text
        return self._simulate(task)

    def _simulate(self, task: str) -> str:
        """Realistic simulated responses per agent type."""
        t = task.lower()
        name = self.name.lower()
        if "bgp" in name or "routing" in name:
            return (
                "BGP DIAGNOSIS: Neighbor 10.0.0.2 (AS 65002) in Active state.\n"
                "Root cause: TCP/179 blocked — firewall rule FW-POLICY-301 denies BGP.\n"
                "Severity: CRITICAL\n"
                "Action: Allow TCP/179 between 10.0.0.1 and 10.0.0.2 on firewall."
            )
        elif "security" in name or "threat" in name:
            return (
                "SECURITY ANALYSIS: 847 failed SSH logins from 203.0.113.45 in 10 min.\n"
                "Pattern: credential brute force — dictionary attack signature.\n"
                "Severity: HIGH\n"
                "Action: Block 203.0.113.45 on edge ACL, enable SSH rate-limiting."
            )
        elif "performance" in name or "metrics" in name:
            return (
                "PERFORMANCE ANALYSIS: Router-Core-01 CPU at 91%, Memory at 87%.\n"
                "Root cause: BGP full table recalculation after peer flap.\n"
                "Severity: HIGH\n"
                "Action: Reduce BGP scan-time, enable route dampening."
            )
        elif "config" in name or "actor" in name:
            return (
                "interface GigabitEthernet0/0\n"
                " ip address 10.0.0.1 255.255.255.0\n"
                " no shutdown\n"
                "router bgp 65001\n"
                " neighbor 10.0.0.2 remote-as 65002\n"
                " neighbor 10.0.0.2 password ENCRYPTED_SECRET\n"
                " neighbor 10.0.0.2 timers 10 30\n"
                "ip access-list extended BGP-IN\n"
                " permit tcp host 10.0.0.2 any eq 179"
            )
        elif "critic" in name or "review" in name:
            return (
                "REVIEW RESULT: APPROVED\n"
                "Checks passed: password encrypted ✓, shutdown avoided ✓, timers valid ✓\n"
                "No security policy violations detected.\n"
                "Risk level: LOW — safe to proceed to change window."
            )
        elif "supervisor" in name or "coordinator" in name:
            if "bgp" in t or "neighbor" in t or "routing" in t:
                return "ROUTE_TO: bgp_agent"
            elif "login" in t or "attack" in t or "brute" in t or "ssh" in t:
                return "ROUTE_TO: security_agent"
            elif "cpu" in t or "memory" in t or "utilization" in t or "performance" in t:
                return "ROUTE_TO: performance_agent"
            return "ROUTE_TO: bgp_agent"
        return f"[{self.name}] Task processed: {task[:60]}..."

print("✓ Setup complete — Agent base class ready")
print(f"  Mode: {'Real API' if USE_REAL_API else 'Simulation'}")


## Demo 1: Supervisor Pattern (Hub-and-Spoke)

**The Problem**: Network alerts arrive in mixed form — some are BGP issues, some are security threats, some are performance degradations. Each needs a different specialist.

**The Solution**: A lightweight Supervisor agent reads the alert, decides which specialist to call, and routes the task.

### Architecture
```
                    ┌─────────────┐
Alert arrives  ───▶ │  Supervisor  │ (reads alert, routes to specialist)
                    └──────┬──────┘
                           │
            ┌──────────────┼──────────────┐
            ▼              ▼              ▼
     ┌─────────────┐ ┌──────────┐ ┌─────────────┐
     │  BGP Agent  │ │ Security │ │ Performance │
     │(routing exp)│ │  Agent   │ │    Agent    │
     └─────────────┘ └──────────┘ └─────────────┘
```

### Key Design Decisions
- Supervisor is **lightweight** — it only classifies and routes, no domain knowledge needed
- Specialists have **focused context** — 3,000 tokens of domain info > 15,000 mixed tokens
- Easy to **add new specialists** without touching the supervisor logic


In [ ]:
# ── Demo 1: Supervisor Pattern ────────────────────────────────────────────────

# ── Specialist agents ─────────────────────────────────────────────────────────
bgp_agent = Agent(
    name="bgp_agent",
    system=(
        "You are a BGP routing specialist. Diagnose BGP neighbor issues, "
        "session state problems, and routing table anomalies. "
        "Always identify: root cause, severity (CRITICAL/HIGH/MEDIUM), and action."
    )
)

security_agent = Agent(
    name="security_agent",
    system=(
        "You are a network security specialist. Analyze security threats, "
        "intrusion attempts, and policy violations. "
        "Always identify: threat type, severity (CRITICAL/HIGH/MEDIUM), and action."
    )
)

performance_agent = Agent(
    name="performance_agent",
    system=(
        "You are a network performance specialist. Diagnose CPU, memory, "
        "bandwidth, and latency issues on network devices. "
        "Always identify: root cause, severity, and action."
    )
)

# ── Specialist registry ───────────────────────────────────────────────────────
AGENTS = {
    "bgp_agent":         bgp_agent,
    "security_agent":    security_agent,
    "performance_agent": performance_agent,
}

# ── Supervisor agent ──────────────────────────────────────────────────────────
supervisor = Agent(
    name="supervisor",
    system=(
        "You are a NOC supervisor. Read each alert and decide which specialist to call.\n"
        "Reply with ONLY one of these exact strings:\n"
        "  ROUTE_TO: bgp_agent\n"
        "  ROUTE_TO: security_agent\n"
        "  ROUTE_TO: performance_agent\n"
        "Nothing else — just the routing decision."
    )
)

def route_alert(alert: str) -> dict:
    """
    Full supervisor routing flow:
    1. Supervisor reads alert → decides which specialist
    2. Specialist receives alert → produces diagnosis
    """
    # Step 1: Supervisor routing
    routing = supervisor.run(alert)

    # Parse routing decision
    chosen_agent = None
    for agent_name in AGENTS:
        if agent_name in routing.lower():
            chosen_agent = agent_name
            break

    if not chosen_agent:
        chosen_agent = "bgp_agent"   # safe fallback

    # Step 2: Specialist analysis
    specialist = AGENTS[chosen_agent]
    diagnosis  = specialist.run(alert)

    return {
        "alert":     alert[:80],
        "routed_to": chosen_agent,
        "diagnosis": diagnosis,
    }

# ── Test with 3 different alert types ─────────────────────────────────────────
alerts = [
    "BGP neighbor 10.0.0.2 (AS 65002) has been in Active state for 8 minutes on Edge-Router-01.",
    "847 failed SSH login attempts from IP 203.0.113.45 in the last 10 minutes.",
    "Router-Core-01 CPU utilization at 91% for the past 15 minutes. SNMP trap received.",
]

print("SUPERVISOR PATTERN — Hub-and-Spoke Routing")
print("=" * 60)

for alert in alerts:
    print(f"\n📨 Alert: {alert[:70]}...")
    result = route_alert(alert)
    print(f"   ▶ Supervisor routed to: {result['routed_to']}")
    print(f"   ▶ Specialist diagnosis:")
    for line in result['diagnosis'].split("\n"):
        print(f"     {line}")

print()
print("=" * 60)
print("KEY INSIGHT:")
print("  Supervisor stays lightweight — just classify + route")
print("  Each specialist has focused, domain-relevant context")
print("  Add a new specialist? Just add to AGENTS dict — supervisor adapts")


## Demo 2: Actor-Critic Pattern — Self-Review Before Production

**The Problem**: A single agent generating a Cisco config change can produce plausible-looking but wrong or unsafe configs — and nobody checks it before it goes to the router.

**The Solution**: Two-agent loop. Actor generates the config. Critic evaluates it against safety rules. If rejected, Actor revises. Loop until Critic approves (or max retries hit).

### How It Works
```
      ┌──────────────────────────────────────────┐
      │           ACTOR-CRITIC LOOP              │
      │                                          │
      │  [Actor]  generates config               │
      │     ↓                                    │
      │  [Critic] evaluates against rules        │
      │     ↓                                    │
      │   PASS → done ✅                         │
      │   FAIL → Actor gets feedback, revises    │
      │     ↓                                    │
      │   Repeat (max 3 iterations)              │
      └──────────────────────────────────────────┘
```

### Critic's Checklist (for network configs)
- No plaintext passwords (`password 0` or `enable password`)
- No `shutdown` on required interfaces
- BGP timers in valid range (keepalive < holdtime)
- Required security features present (SSH version 2, no telnet)

> **Network Analogy**: Actor-Critic = dual-control principle in change management. One engineer writes the change (Actor), a second reviews it against the change policy (Critic). The Critic doesn't redesign — just checks the safety checklist.


In [ ]:
# ── Demo 2: Actor-Critic Pattern ──────────────────────────────────────────────

actor_agent = Agent(
    name="config_actor",
    system=(
        "You are a network configuration engineer. Generate Cisco IOS configurations. "
        "Follow best practices: encrypt passwords, use SSH v2, set BGP timers correctly, "
        "never use 'shutdown' on required interfaces."
    )
)

critic_agent = Agent(
    name="config_critic",
    system=(
        "You are a network security reviewer. Check Cisco IOS configs against policy.\n"
        "RULES (check all):\n"
        "  1. No plaintext passwords (no 'password 0' or 'enable password')\n"
        "  2. No 'shutdown' on production interfaces\n"
        "  3. BGP keepalive < holdtime (e.g., keepalive 10, holdtime 30 is OK)\n"
        "  4. SSH version 2 only\n"
        "Reply with APPROVED or REJECTED.\n"
        "If REJECTED, list exactly which rules failed."
    )
)

def actor_critic_loop(task: str, max_iterations: int = 3) -> dict:
    """
    Actor generates config → Critic reviews → loop until approved or max retries.
    Returns the approved config or best attempt with rejection reason.
    """
    feedback = ""
    history  = []

    for iteration in range(1, max_iterations + 1):
        print(f"\n  Iteration {iteration}/{max_iterations}")

        # Actor generates (or revises based on Critic feedback)
        actor_prompt = task
        if feedback:
            actor_prompt += f"\n\nCritic rejected previous version because:\n{feedback}\nFix these issues."
        config = actor_agent.run(actor_prompt)
        print(f"  [Actor]  Generated config ({len(config)} chars)")

        # Critic evaluates
        critic_prompt = f"Review this Cisco IOS configuration:\n\n{config}"
        review = critic_agent.run(critic_prompt)
        approved = "APPROVED" in review.upper()
        print(f"  [Critic] Result: {'✅ APPROVED' if approved else '❌ REJECTED'}")

        history.append({
            "iteration": iteration,
            "config":    config,
            "review":    review,
            "approved":  approved,
        })

        if approved:
            print(f"  ✅ Config approved after {iteration} iteration(s)")
            return {"status": "approved", "config": config,
                    "iterations": iteration, "history": history}

        # Extract feedback for next Actor iteration
        feedback = review.replace("REJECTED", "").strip()
        if not feedback:
            feedback = "Did not pass safety review. Ensure passwords are encrypted, SSH v2 is set."

    print(f"  ⚠ Max iterations ({max_iterations}) reached — escalate to human review")
    return {"status": "escalated", "config": history[-1]["config"],
            "iterations": max_iterations, "history": history}

# ── Run demo ──────────────────────────────────────────────────────────────────
TASK = (
    "Generate a Cisco IOS config to establish a BGP peering session with neighbor "
    "10.0.0.2 (AS 65002). Include: interface IP, BGP config with timers (keepalive 10, "
    "holdtime 30), encrypted neighbor password, and an ACL allowing TCP/179."
)

print("ACTOR-CRITIC PATTERN — Self-Review Loop")
print("=" * 60)
print(f"Task: {TASK[:80]}...")

result = actor_critic_loop(TASK, max_iterations=3)

print()
print("FINAL RESULT:")
print(f"  Status:     {result['status'].upper()}")
print(f"  Iterations: {result['iterations']}")
print()
print("APPROVED CONFIG:")
print("-" * 40)
print(result['config'])
print()
print("ACTOR-CRITIC BENEFITS:")
print("  ✓ Config reviewed against safety checklist before production")
print("  ✓ Actor self-corrects based on Critic feedback")
print("  ✓ max_iterations prevents infinite loops")
print("  ✓ Escalates to human if not resolved — no silent failures")


## Demo 3: Sequential vs Parallel Agent Execution

**The Problem**: Running 3 specialist agents one after another (sequential) takes 3× as long as needed if the agents don't depend on each other's outputs.

**The Solution**: Run independent agents in parallel using threads. Only run sequentially when agent B genuinely needs agent A's output.

### Latency Comparison
```
SEQUENTIAL (agents depend on each other):
  Agent A: 3s ──▶ Agent B: 3s ──▶ Agent C: 3s = 9s total

PARALLEL (agents are independent):
  Agent A: 3s ─────────┐
  Agent B: 3s ──────────┼──▶ merge = ~3s total
  Agent C: 3s ─────────┘
```

### Decision Rule: When to Parallelize?

| Scenario | Pattern | Reason |
|----------|---------|--------|
| BGP diagnosis + Security scan (same alert, independent) | **Parallel** | Neither needs the other's output |
| Classify → Extract → Runbook | **Sequential** | Each step needs the previous output |
| Multi-domain fault investigation | **Parallel first, synthesize second** | Hybrid |


In [ ]:
# ── Demo 3: Sequential vs Parallel Execution ──────────────────────────────────
import threading
import time

def timed_agent_run(agent: Agent, task: str, results: dict, key: str):
    """Run an agent and store result + elapsed time in shared dict."""
    start  = time.time()
    result = agent.run(task)
    elapsed = time.time() - start
    results[key] = {"output": result, "elapsed": elapsed}

# ── Agents for this demo ──────────────────────────────────────────────────────
routing_agent = Agent(
    name="routing_agent",
    system="You are a routing specialist. Analyze BGP and OSPF issues in 3 sentences max."
)
security_scan_agent = Agent(
    name="security_agent",
    system="You are a security analyst. Check for ACL violations and attack patterns in 3 sentences max."
)
metrics_agent = Agent(
    name="performance_agent",
    system="You are a performance engineer. Analyze CPU/memory/bandwidth metrics in 3 sentences max."
)

ALERT = (
    "Core-Router-01: BGP neighbor down (AS 65002), CPU at 89%, "
    "and 200 failed auth attempts detected in the last 5 minutes."
)

# ── SEQUENTIAL run ────────────────────────────────────────────────────────────
print("SEQUENTIAL EXECUTION (one after another):")
print("-" * 50)
seq_results = {}
seq_start = time.time()

for agent, key in [(routing_agent, "routing"), (security_scan_agent, "security"), (metrics_agent, "performance")]:
    t = time.time()
    seq_results[key] = agent.run(ALERT)
    print(f"  {key:12s} done in {time.time()-t:.2f}s")

seq_total = time.time() - seq_start
print(f"  Total time: {seq_total:.2f}s")

# ── PARALLEL run ──────────────────────────────────────────────────────────────
print()
print("PARALLEL EXECUTION (all at once via threads):")
print("-" * 50)
par_results = {}
threads = []
par_start = time.time()

for agent, key in [(routing_agent, "routing"), (security_scan_agent, "security"), (metrics_agent, "performance")]:
    t = threading.Thread(target=timed_agent_run,
                         args=(agent, ALERT, par_results, key))
    threads.append(t)

for t in threads:
    t.start()
for t in threads:
    t.join()

par_total = time.time() - par_start
for key, r in par_results.items():
    print(f"  {key:12s} done in {r['elapsed']:.2f}s")
print(f"  Total time: {par_total:.2f}s")

# ── Comparison ────────────────────────────────────────────────────────────────
print()
print("=" * 50)
print(f"COMPARISON:")
print(f"  Sequential: {seq_total:.2f}s")
print(f"  Parallel:   {par_total:.2f}s")
speedup = seq_total / par_total if par_total > 0 else 1
print(f"  Speedup:    {speedup:.1f}× faster with parallel")
print()
print("COMBINED FINDINGS:")
for key in ["routing", "security", "performance"]:
    result = par_results.get(key, {}).get("output", seq_results.get(key, ""))
    print(f"\n[{key.upper()} AGENT]")
    print(f"  {str(result)[:100]}...")

print()
print("RULE OF THUMB:")
print("  Parallel → when agents work on the SAME input independently")
print("  Sequential → when agent B needs agent A's OUTPUT as its input")


## Demo 4: Shared State + Clean Agent Handoffs

**The Problem**: In a multi-step workflow, how does Agent B know what Agent A already did? Passing raw text between agents loses structure and context.

**The Solution**: A shared state dictionary — the system's working memory. Every agent reads from it and writes back to it. Handoffs transfer state, not just text.

### Shared State as Working Memory
```python
state = {
    "alert":         "BGP neighbor down...",
    "severity":      None,     # filled by Classifier agent
    "diagnosis":     None,     # filled by Specialist agent
    "runbook":       None,     # filled by Runbook agent
    "status":        "pending" # tracks workflow progress
}
```

### Handoff Protocol
1. Agent reads relevant fields from state
2. Agent does its work
3. Agent writes results back to state
4. Agent marks its step as `"done"`
5. Next agent picks up where previous left off

> **Why This Matters**: State validation catches cascades early — if the Classifier wrote `severity = None`, the Specialist knows to refuse and escalate rather than hallucinate a diagnosis on bad data.


In [ ]:
# ── Demo 4: Shared State + Agent Handoffs ─────────────────────────────────────

@dataclass
class WorkflowState:
    """Shared working memory for the multi-agent workflow."""
    alert:        str
    severity:     Optional[str] = None
    routed_to:    Optional[str] = None
    diagnosis:    Optional[str] = None
    runbook:      Optional[str] = None
    status:       str = "pending"
    steps_done:   list = field(default_factory=list)
    errors:       list = field(default_factory=list)

    def update(self, **kwargs):
        """Update state fields and log the step."""
        for k, v in kwargs.items():
            setattr(self, k, v)

    def is_valid_for(self, step: str) -> tuple:
        """Validate state has required data before an agent runs."""
        requirements = {
            "classification": ["alert"],
            "diagnosis":      ["alert", "severity", "routed_to"],
            "runbook":        ["diagnosis"],
        }
        needed = requirements.get(step, [])
        for field_name in needed:
            val = getattr(self, field_name, None)
            if val is None:
                return False, f"Missing '{field_name}' — required for {step}"
        return True, ""

# ── Agents for handoff demo ───────────────────────────────────────────────────
classifier = Agent(
    name="classifier",
    system=(
        "Classify the network alert severity. "
        "Reply with ONLY: CRITICAL, HIGH, MEDIUM, or LOW. Nothing else."
    )
)

specialist = Agent(
    name="specialist",
    system=(
        "You are a network operations specialist. "
        "Diagnose the alert. Identify root cause and recommended action in 4 sentences max."
    )
)

runbook_agent = Agent(
    name="runbook_agent",
    system=(
        "You are a NOC documentation specialist. "
        "Convert a diagnosis into a numbered runbook (max 5 steps) with specific CLI commands."
    )
)

def run_workflow(alert: str) -> WorkflowState:
    """Execute 3-agent workflow with shared state and handoff validation."""
    state = WorkflowState(alert=alert)
    print(f"Starting workflow for: {alert[:65]}...")
    print()

    # ── Step 1: Classifier ────────────────────────────────────────────────
    print("  [Step 1] Classifier agent running...")
    ok, err = state.is_valid_for("classification")
    if not ok:
        state.errors.append(f"Step 1 blocked: {err}")
        state.status = "error"
        return state

    raw_severity = classifier.run(state.alert)
    severity = "CRITICAL"   # default
    for level in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
        if level in raw_severity.upper():
            severity = level
            break

    state.update(severity=severity, routed_to="specialist", status="classified")
    state.steps_done.append("classification")
    print(f"    ✓ Severity: {state.severity} → routing to specialist")

    # ── Step 2: Specialist (reads severity from state) ────────────────────
    print(f"  [Step 2] Specialist agent running (severity={state.severity})...")
    ok, err = state.is_valid_for("diagnosis")
    if not ok:
        state.errors.append(f"Step 2 blocked: {err}")
        state.status = "error"
        return state

    # Specialist gets alert + severity as context (clean handoff)
    context = f"Alert severity: {state.severity}"
    diagnosis = specialist.run(state.alert, context=context)
    state.update(diagnosis=diagnosis, status="diagnosed")
    state.steps_done.append("diagnosis")
    print(f"    ✓ Diagnosis complete ({len(diagnosis)} chars)")

    # ── Step 3: Runbook (reads diagnosis from state) ──────────────────────
    print("  [Step 3] Runbook agent running (reads diagnosis from state)...")
    ok, err = state.is_valid_for("runbook")
    if not ok:
        state.errors.append(f"Step 3 blocked: {err}")
        state.status = "error"
        return state

    runbook = runbook_agent.run(state.diagnosis)
    state.update(runbook=runbook, status="complete")
    state.steps_done.append("runbook")
    print(f"    ✓ Runbook generated")

    return state

# ── Run demo ──────────────────────────────────────────────────────────────────
ALERT = ("BGP neighbor 10.0.0.2 (AS 65002) down on Edge-Router-01. "
         "Routes from AS 65002 withdrawn. Failover to backup path initiated.")

print("SHARED STATE + AGENT HANDOFFS")
print("=" * 60)
state = run_workflow(ALERT)

print()
print("FINAL STATE:")
print(f"  Status:      {state.status}")
print(f"  Steps done:  {' → '.join(state.steps_done)}")
print(f"  Severity:    {state.severity}")
print()
print("DIAGNOSIS:")
print(state.diagnosis)
print()
print("RUNBOOK:")
print(state.runbook)

if state.errors:
    print()
    print("ERRORS CAUGHT:")
    for e in state.errors:
        print(f"  ✗ {e}")


## Demo 5: Hallucination Cascade Defense

**The Problem**: Agent A hallucates wrong data. Agent B trusts it (it looks authoritative — it came from another agent). Agent C builds on B's wrong output. By the time results reach the user, 3 agents have confidently agreed on a false diagnosis.

This is the **hallucination cascade** — worse than a single-agent hallucination because it creates false consensus.

### Example Cascade
```
Agent A: "BGP peer 10.0.0.2 is DOWN due to MTU mismatch" ← hallucinated
Agent B: "Confirmed MTU issue. Recommend changing MTU to 1480 on all interfaces"
Agent C: "Runbook: Step 1: Set MTU 1480 on all interfaces across the network"

Real issue: firewall blocked TCP/179 — nothing to do with MTU
Result: Wrong config change pushed to all routers 💥
```

### Defense: Validation Gate Between Agents

```
Agent A output
    ↓
[Validator] checks:
  • Is the output structured? (has required fields)
  • Is the confidence stated? (below threshold → flag)
  • Does it contradict known facts? (IP exists? protocol valid?)
    ↓
PASS → Agent B receives validated data
FAIL → escalate to human, do NOT pass forward
```


In [ ]:
# ── Demo 5: Hallucination Cascade Defense ─────────────────────────────────────

class ValidationGate:
    """
    Sits between agents. Validates output before it is passed downstream.
    Catches structural issues, low confidence, and implausible data.
    """

    # Structured output must contain these fields (case-insensitive)
    REQUIRED_KEYWORDS = ["root cause", "severity", "action"]

    # Known valid network protocols
    VALID_PROTOCOLS = {"BGP", "OSPF", "EIGRP", "STP", "LLDP", "CDP",
                       "SSH", "SNMP", "NTP", "HTTPS", "TCP", "UDP"}

    def __init__(self, min_confidence: int = 50):
        self.min_confidence = min_confidence
        self.checks_run     = 0
        self.blocks_issued  = 0

    def validate(self, output: str, source_agent: str) -> dict:
        """
        Run all validation checks.
        Returns: {valid, reasons, warnings}
        """
        self.checks_run += 1
        reasons  = []
        warnings = []

        # Check 1: Minimum length (too short = likely incomplete)
        if len(output.strip()) < 30:
            reasons.append("Output too short — likely incomplete response")

        # Check 2: Required structure keywords present
        output_lower = output.lower()
        missing = [kw for kw in self.REQUIRED_KEYWORDS
                   if kw.lower() not in output_lower]
        if missing:
            warnings.append(f"Missing sections: {missing} — may be incomplete")

        # Check 3: Confidence extraction (look for "confidence: XX%")
        import re
        conf_match = re.search(r'confidence[:\s]+([0-9]+)%?', output_lower)
        if conf_match:
            conf = int(conf_match.group(1))
            if conf < self.min_confidence:
                reasons.append(
                    f"Confidence {conf}% below threshold {self.min_confidence}% "
                    f"— do not auto-act on this diagnosis"
                )

        # Check 4: Implausible protocol references
        import re as _re
        found_protocols = set(_re.findall(r'\b([A-Z]{2,8})\b', output))
        suspicious = found_protocols - self.VALID_PROTOCOLS - {
            "CPU", "RAM", "CLI", "ACL", "QOS", "MTU", "ARP", "DNS",
            "VLAN", "MPLS", "VPN", "BGP", "NOC", "NOC", "IOS", "ASN",
            "CRITICAL", "HIGH", "MEDIUM", "LOW", "OK", "UP", "DOWN",
            "IP", "LAN", "WAN", "ISP", "SLA", "NMS", "IETF", "RFC"
        }
        if len(suspicious) > 3:
            warnings.append(f"Unusual acronyms: {list(suspicious)[:5]} — verify accuracy")

        # Check 5: Empty or error-like output
        error_phrases = ["i cannot", "i don't know", "unable to", "no information"]
        if any(p in output_lower for p in error_phrases):
            reasons.append("Agent expressed uncertainty — do not pass downstream")

        valid = len(reasons) == 0
        if not valid:
            self.blocks_issued += 1

        return {
            "valid":        valid,
            "source_agent": source_agent,
            "reasons":      reasons,
            "warnings":     warnings,
        }

def guarded_pipeline(alert: str) -> dict:
    """
    3-agent pipeline with validation gates between each agent.
    Any blocked step stops the cascade immediately.
    """
    gate = ValidationGate(min_confidence=60)
    results = {"alert": alert, "steps": [], "final_runbook": None, "cascade_blocked": False}

    print(f"Alert: {alert[:70]}...")
    print()

    # Specialist agents
    diag_agent   = Agent("bgp_agent",      "You are a BGP specialist. Diagnose with root cause, severity, and action.")
    enrich_agent = Agent("enricher_agent", "You are a context enricher. Add relevant BGP troubleshooting commands.")
    runbook_builder = Agent("runbook_agent", "Convert this diagnosis into a 5-step NOC runbook with CLI commands.")

    agents_tasks = [
        (diag_agent,   alert,       "Step 1 — Diagnosis"),
        (enrich_agent, None,        "Step 2 — Enrichment"),  # uses previous output
        (runbook_builder, None,     "Step 3 — Runbook"),     # uses previous output
    ]

    previous_output = None

    for i, (agent, task, label) in enumerate(agents_tasks):
        print(f"  [{label}]")
        task_input = task if task else previous_output

        output = agent.run(task_input)

        # Run validation gate
        check = gate.validate(output, agent.name)
        results["steps"].append({
            "agent":  agent.name,
            "output": output[:100],
            "valid":  check["valid"],
        })

        if check["warnings"]:
            for w in check["warnings"]:
                print(f"    ⚠ Warning: {w}")

        if not check["valid"]:
            print(f"    🚫 GATE BLOCKED: {'; '.join(check['reasons'])}")
            print(f"    → Stopping cascade — escalating to human review")
            results["cascade_blocked"] = True
            break
        else:
            print(f"    ✅ Validation passed — forwarding to next agent")
            previous_output = output

    if not results["cascade_blocked"]:
        results["final_runbook"] = previous_output

    return results, gate

# ── Test: good alert vs bad alert ─────────────────────────────────────────────
print("HALLUCINATION CASCADE DEFENSE")
print("=" * 60)

# Good alert — agents should produce valid output
r, gate = guarded_pipeline(
    "BGP neighbor 10.0.0.2 (AS 65002) in Active state. "
    "TCP/179 connectivity confirmed. Keepalive not received for 45s."
)

print()
if r["cascade_blocked"]:
    print("Pipeline BLOCKED — escalated to human (no bad data passed forward)")
else:
    print("Pipeline COMPLETE — all agents validated")
    print()
    print("FINAL RUNBOOK:")
    print(r["final_runbook"])

print()
print("=" * 60)
print(f"GATE STATISTICS:")
print(f"  Checks run:    {gate.checks_run}")
print(f"  Blocks issued: {gate.blocks_issued}")
print()
print("HALLUCINATION CASCADE DEFENSE RULES:")
print("  ✓ Never trust inter-agent messages without validation")
print("  ✓ Check for required structure (root cause, severity, action)")
print("  ✓ Low confidence → stop and escalate, don't pass forward")
print("  ✓ Uncertain agent output → human review, not automated action")


## Summary — Which Pattern for Which Network Task

| Task | Pattern | Why |
|------|---------|-----|
| Mixed-domain alert triage | **Supervisor** | Lightweight routing to the right specialist |
| Config generation before production | **Actor-Critic** | Self-review catches issues before commit |
| Independent multi-domain analysis | **Parallel Execution** | 3× speedup when agents don't need each other |
| Sequential diagnosis workflow | **Sequential + Shared State** | Clean handoffs with state validation |
| Any automated multi-agent pipeline | **Validation Gates** | Prevents hallucination cascades |

## The Three Rules for Safe Multi-Agent Systems

1. **Always set max_iterations** — without it, agents can loop indefinitely
2. **Validate between agents** — treat agent-to-agent messages like user input (not trusted)
3. **Human-in-the-Loop for irreversible actions** — no agent should push config changes without approval

## Common Failure Modes to Watch

| Failure | Cause | Prevention |
|---------|-------|------------|
| Hallucination cascade | Agent B trusts Agent A blindly | Validation gates |
| Infinite loop | No termination condition | `max_iterations` everywhere |
| State divergence | Two agents write conflicting data | Sequential writes, validate on read |
| Specification gap | Overlapping or missing agent roles | Define clear role boundaries |


In [ ]:
# ── Full Integration: NOC Multi-Agent System ─────────────────────────────────
# Combines: Supervisor + Parallel Execution + Shared State + Validation Gates

class NOCMultiAgentSystem:
    """
    Complete NOC alert processing system using 4 patterns together.

    Flow:
      1. Supervisor classifies + routes (Supervisor pattern)
      2. Routing + Security agents run in parallel (Parallel execution)
      3. Synthesis agent combines results (Sequential + Shared State)
      4. Validation gate before runbook generation (Cascade defense)
    """

    def __init__(self):
        self.supervisor   = Agent("supervisor", "Classify network alerts. Reply ONLY: ROUTE_TO: bgp_agent, security_agent, or performance_agent")
        self.bgp_agent    = Agent("bgp_agent",  "BGP/routing specialist. Diagnose with root cause, severity (CRITICAL/HIGH/MEDIUM), action.")
        self.sec_agent    = Agent("security_agent", "Security specialist. Analyze threats with severity and action.")
        self.perf_agent   = Agent("performance_agent", "Performance specialist. Diagnose CPU/memory/bandwidth issues with severity and action.")
        self.synthesizer  = Agent("synthesizer", "Synthesize multiple specialist reports into a unified incident summary.")
        self.gate         = ValidationGate(min_confidence=55)

    def process(self, alert: str) -> dict:
        result = {"alert": alert[:80], "parallel_findings": {}, "synthesis": None,
                  "runbook": None, "blocked": False, "agents_used": []}

        # ── Step 1: Supervisor routing ──────────────────────────────────
        routing = self.supervisor.run(alert)
        primary = "bgp_agent"
        for a in ["bgp_agent", "security_agent", "performance_agent"]:
            if a in routing.lower():
                primary = a
                break
        result["agents_used"].append(f"supervisor→{primary}")

        # ── Step 2: Parallel specialist execution ───────────────────────
        # Always run BGP + Security in parallel for comprehensive view
        par_results = {}
        threads = [
            threading.Thread(target=timed_agent_run, args=(self.bgp_agent, alert, par_results, "bgp")),
            threading.Thread(target=timed_agent_run, args=(self.sec_agent, alert, par_results, "security")),
        ]
        for t in threads: t.start()
        for t in threads: t.join()

        result["parallel_findings"] = {k: v["output"] for k, v in par_results.items()}
        result["agents_used"].extend(["bgp_agent(parallel)", "security_agent(parallel)"])

        # ── Step 3: Validate each finding before synthesis ──────────────
        for key, finding in result["parallel_findings"].items():
            check = self.gate.validate(finding, f"{key}_agent")
            if not check["valid"]:
                result["blocked"] = True
                result["block_reason"] = f"{key}: {check['reasons'][0]}"
                return result

        # ── Step 4: Synthesis (sequential — needs parallel findings) ────
        combined = "\n\n".join([
            f"[BGP SPECIALIST]\n{result['parallel_findings']['bgp']}",
            f"[SECURITY SPECIALIST]\n{result['parallel_findings']['security']}",
        ])
        synthesis = self.synthesizer.run(
            "Synthesize these specialist reports into a unified incident summary:",
            context=combined
        )
        result["synthesis"] = synthesis
        result["agents_used"].append("synthesizer(sequential)")

        # ── Step 5: Final runbook with validation gate ──────────────────
        check = self.gate.validate(synthesis, "synthesizer")
        if not check["valid"]:
            result["blocked"] = True
            result["block_reason"] = f"Synthesis validation: {check['reasons'][0]}"
            return result

        runbook_builder = Agent("runbook_agent",
            "Create a 5-step NOC runbook with CLI commands from this incident summary.")
        result["runbook"] = runbook_builder.run(synthesis)
        result["agents_used"].append("runbook_agent")

        return result

# ── Test the full system ───────────────────────────────────────────────────────
noc = NOCMultiAgentSystem()

ALERT = (
    "CRITICAL INCIDENT: Edge-Router-01 — BGP neighbor 10.0.0.2 (AS 65002) down, "
    "CPU at 89%, and 500 failed SSH auth attempts detected. "
    "Customer traffic to AS 65002 is black-holing."
)

print("NOC MULTI-AGENT SYSTEM — Full Integration")
print("=" * 65)
print(f"Alert: {ALERT}")
print()

result = noc.process(ALERT)

if result["blocked"]:
    print(f"🚫 Pipeline BLOCKED: {result.get('block_reason', 'validation failed')}")
    print("   → Escalated to human NOC engineer")
else:
    print(f"Agents used: {' → '.join(result['agents_used'])}")
    print()
    print("PARALLEL FINDINGS:")
    for domain, finding in result["parallel_findings"].items():
        print(f"  [{domain.upper()}]: {str(finding)[:80]}...")
    print()
    print("UNIFIED SYNTHESIS:")
    print(result["synthesis"])
    print()
    print("NOC RUNBOOK:")
    print(result["runbook"])

print()
print("=" * 65)
print("PATTERNS COMBINED:")
print("  1. Supervisor    → classified + routed the alert")
print("  2. Parallel      → BGP + Security ran simultaneously")
print("  3. Shared State  → findings passed cleanly to synthesizer")
print("  4. Validation    → gates blocked hallucinations before runbook")
print()
print("✅ Chapter 34 complete — Multi-Agent Orchestration mastered!")
